# Geospatial POI and Routing Features

## Objective

This notebook develops external geographic features for Victorian
rental properties, including:

- distance to nearest train station
- distance to Melbourne CBD
- proximity to schools
- proximity to parks
- proximity to shopping/amenities

Straight-line distance is used as an initial baseline.
OpenRouteService route distance will subsequently be calculated
at suburb level to reduce API usage.

In [35]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.neighbors import BallTree

## 1. File paths

Define the locations of the raw and curated datasets.

In [36]:
PROJECT_ROOT = Path("..")

RAW_DIR = PROJECT_ROOT / "data" / "raw"
CURATED_DIR = PROJECT_ROOT / "data" / "curated"

TRANSPORT_DIR = RAW_DIR / "transport"
SCHOOL_DIR = RAW_DIR / "schools"
OSM_DIR = RAW_DIR / "osm"

CURATED_DIR.mkdir(parents=True, exist_ok=True)

## 2. Victorian train stations

Transport Victoria GTFS data are divided by transport mode.

The metropolitan train dataset and regional train dataset are combined
so that train accessibility can be calculated for properties throughout Victoria.

Only station-level locations are retained where possible, rather than
individual platforms or entrances.

In [37]:
metro_train_path = (
    TRANSPORT_DIR
    / "gtfs"
    / "1"
    / "google_transit"
    / "stops.txt"
)

regional_train_path = (
    TRANSPORT_DIR
    / "gtfs"
    / "2"
    / "google_transit"
    / "stops.txt"
)

metro_stops = pd.read_csv(metro_train_path)
regional_stops = pd.read_csv(regional_train_path)

print("Metro stop records:", len(metro_stops))
print("Regional stop records:", len(regional_stops))

Metro stop records: 627
Regional stop records: 2859


In [38]:
train_stops = pd.concat(
    [metro_stops, regional_stops],
    ignore_index=True
)

train_stops.head()

,stop_id,stop_name,stop_lat,stop_lon,stop_url,location_type,parent_station,wheelchair_boarding,level_id,platform_code
0,11212,Flinders Street Station,-37.818095,144.966266,https://transport.vic.gov.au/stop/1071/?utm_so...,NaN,vic:rail:FSS,1.0,Level 0,NaN
1,11213,Flinders Street Station,-37.818144,144.966492,https://transport.vic.gov.au/stop/1071/?utm_so...,NaN,vic:rail:FSS,1.0,Level 0,NaN
2,11214,Flinders Street Station,-37.818198,144.966524,https://transport.vic.gov.au/stop/1071/?utm_so...,NaN,vic:rail:FSS,1.0,Level 0,NaN
3,11215,Flinders Street Station,-37.818289,144.966556,https://transport.vic.gov.au/stop/1071/?utm_so...,NaN,vic:rail:FSS,1.0,Level 0,NaN
4,11216,Flinders Street Station,-37.818326,144.966600,https://transport.vic.gov.au/stop/1071/?utm_so...,NaN,vic:rail:FSS,1.0,Level 0,NaN


In [39]:
if "location_type" in train_stops.columns:
    stations = train_stops[
        train_stops["location_type"] == 1
    ].copy()
else:
    stations = train_stops.copy()


stations = stations[
    [
        "stop_id",
        "stop_name",
        "stop_lat",
        "stop_lon"
    ]
].copy()

stations = stations.rename(columns={
    "stop_lat": "latitude",
    "stop_lon": "longitude"
})


stations = stations.dropna(
    subset=["latitude", "longitude"]
)

stations = stations.drop_duplicates(
    subset=[
        "stop_name",
        "latitude",
        "longitude"
    ]
).reset_index(drop=True)

print("Number of train stations:", len(stations))

stations.head(10)


Number of train stations: 320


,stop_id,stop_name,latitude,longitude
0,nsw:rail:ABY,Albury Railway Station,-36.085046,146.924381
1,vic:rail:ARR,Ardeer Railway Station,-37.783065,144.802188
2,vic:rail:ART,Ararat Railway Station,-37.282241,142.936912
3,vic:rail:AVL,Avenel Railway Station,-36.893648,145.229515
4,vic:rail:BAH,Bacchus Marsh Railway Station,-37.687578,144.436785
5,vic:rail:BAT-V,Ballarat Railway Station,-37.558791,143.859457
6,vic:rail:BDE,Bairnsdale Railway Station,-37.828720,147.627614
7,vic:rail:BET,Beaufort Railway Station,-37.427627,143.382381
8,vic:rail:BEW,Berwick Railway Station,-38.039980,145.345417
9,vic:rail:BGE,Birregurra Railway Station,-38.328808,143.783625


## 3. Victorian school locations

The Victorian School Locations dataset contains primary and secondary
school locations throughout Victoria.

School coordinates are used to calculate:

- distance to the nearest school
- number of schools within 2 km

These variables are used as measures of access to education and local
liveability.

In [40]:
school_path = (
    SCHOOL_DIR
    / "dv402-SchoolLocations2025.csv"
)

schools = pd.read_csv(school_path)

print("Number of school records:", len(schools))

schools.head()

Number of school records: 2301


,Education_Sector,Entity_Type,School_No,School_Name,School_Type,School_Status,Address_Line_1,Address_Line_2,Address_Town,Address_State,...,Postal_State,Postal_Postcode,Full_Phone_No,Region,Area,LGA_ID,LGA_Name,LGA_TYPE,X,Y
0,Catholic,2,20,Parade College,Secondary,O,1436 Plenty Road,NaN,BUNDOORA,VIC,...,VIC,3083,03 9468 3300,NORTH-WESTERN VICTORIA,North Eastern Melbourne,66,Banyule (C),Metro,145.066978,-37.690178
1,Catholic,2,25,Simonds Catholic College,Secondary,O,273 Victoria Street,NaN,WEST MELBOURNE,VIC,...,VIC,3003,03 9321 9200,SOUTH-WESTERN VICTORIA,Western Melbourne,460,Melbourne (C),Metro,144.952883,-37.805971
2,Catholic,2,26,St Mary’s College Melbourne,Secondary,O,11 Westbury Street,NaN,ST KILDA EAST,VIC,...,VIC,3182,03 9529 6611,SOUTH-EASTERN VICTORIA,Bayside Peninsula,590,Port Phillip (C),Metro,144.997001,-37.859365
3,Catholic,2,28,St Patrick's College Ballarat,Secondary,O,1431 Sturt Street,NaN,BALLARAT,VIC,...,VIC,3350,03 5331 1688,SOUTH-WESTERN VICTORIA,Central Highlands,57,Ballarat (C),Non Metro,143.831558,-37.559711
4,Catholic,2,29,St Patrick's School,Primary,O,119 Drummond Street South,NaN,BALLARAT,VIC,...,VIC,3350,03 5332 7680,SOUTH-WESTERN VICTORIA,Central Highlands,57,Ballarat (C),Non Metro,143.847147,-37.564397


In [41]:
schools = schools[
    [
        "School_No",
        "School_Name",
        "School_Type",
        "Education_Sector",
        "Address_Town",
        "Address_Postcode",
        "X",
        "Y"
    ]
].copy()

schools = schools.rename(columns={
    "X": "longitude",
    "Y": "latitude"
})

schools = schools.dropna(
    subset=["latitude", "longitude"]
).reset_index(drop=True)


## 4. Temporary locations for pipeline development

A small set of Victorian suburbs is used to test the geographic feature
pipeline before the group's complete rental property dataset is available.

These coordinates are temporary development values and will later be
replaced with property coordinates or suburb centroids.

In [42]:
test_locations = pd.DataFrame({
    "suburb": [
        "Carlton",
        "Richmond",
        "Footscray",
        "Box Hill",
        "Werribee"
    ],

    "latitude": [
        -37.800,
        -37.818,
        -37.802,
        -37.819,
        -37.900
    ],

    "longitude": [
        144.967,
        145.002,
        144.900,
        145.122,
        144.661
    ]
})

test_locations

,suburb,latitude,longitude
0,Carlton,-37.800,144.967
1,Richmond,-37.818,145.002
2,Footscray,-37.802,144.900
3,Box Hill,-37.819,145.122
4,Werribee,-37.900,144.661


## 5. Straight-line distance to points of interest

Nearest points of interest are identified using a BallTree with the
Haversine distance metric.

This provides an efficient measure of straight-line geographic distance
between suburb/property locations and external amenities.

In [43]:
EARTH_RADIUS_KM = 6371.0088

def add_nearest_poi(
    locations,
    pois,
    poi_name=None,
    prefix="poi",
    keep_coordinates=False
):

    locations = locations.copy()
    pois = pois.copy()

    location_coords = np.radians(
        locations[
            ["latitude", "longitude"]
        ].to_numpy()
    )

    poi_coords = np.radians(
        pois[
            ["latitude", "longitude"]
        ].to_numpy()
    )

    tree = BallTree(
        poi_coords,
        metric="haversine"
    )

    distances, indices = tree.query(
        location_coords,
        k=1
    )

    nearest_indices = indices[:, 0]

    locations[f"{prefix}_distance_km"] = (
        distances[:, 0]
        * EARTH_RADIUS_KM
    )

    if poi_name is not None:
        locations[f"nearest_{prefix}"] = (
            pois.iloc[nearest_indices][poi_name]
            .to_numpy()
        )

    if keep_coordinates:
        locations[f"{prefix}_latitude"] = (
            pois.iloc[nearest_indices]["latitude"]
            .to_numpy()
        )

        locations[f"{prefix}_longitude"] = (
            pois.iloc[nearest_indices]["longitude"]
            .to_numpy()
        )

    return locations

## 6. Distance to nearest train station

For each location, the closest metropolitan or regional train station
is identified.

Straight-line distance is used initially as the baseline accessibility
measure.

In [44]:
test_locations = add_nearest_poi(
    locations=test_locations,
    pois=stations,
    poi_name="stop_name",
    prefix="train",
    keep_coordinates=True
)

test_locations[
    [
        "suburb",
        "nearest_train",
        "train_distance_km",
        "train_latitude",
        "train_longitude"
    ]
]

,suburb,nearest_train,train_distance_km,train_latitude,train_longitude
0,Carlton,Parkville Railway Station,0.655380,-37.799874,144.959542
1,Richmond,West Richmond Railway Station,0.989123,-37.814949,144.991423
2,Footscray,Footscray Railway Station,0.189086,-37.801413,144.902020
3,Box Hill,Box Hill Railway Station,0.055890,-37.819222,145.121429
4,Werribee,Werribee Railway Station,0.069929,-37.899378,144.661118


## 7. Distance to nearest school

The closest Victorian school is identified for each location.

This feature measures access to nearby educational facilities.

In [45]:
test_locations = add_nearest_poi(
    locations=test_locations,
    pois=schools,
    poi_name="School_Name",
    prefix="school"
)

test_locations[
    [
        "suburb",
        "nearest_school",
        "school_distance_km"
    ]
]

,suburb,nearest_school,school_distance_km
0,Carlton,Carlton Gardens Primary School,0.317048
1,Richmond,Richmond High School,0.096545
2,Footscray,Footscray City Primary School,0.489763
3,Box Hill,Our Lady of Sion College,0.699279
4,Werribee,Wyndham Community and Education Centre Inc | J...,0.247769


## 8. Number of nearby points of interest

In addition to nearest-distance measures, amenities within a specified
radius are counted.

For schools, a 2 km radius is used initially as a measure of local
educational accessibility.

In [46]:
def count_pois_within_radius(
    locations,
    pois,
    radius_km,
    prefix
):

    locations = locations.copy()

    location_coords = np.radians(
        locations[
            ["latitude", "longitude"]
        ].to_numpy()
    )

    poi_coords = np.radians(
        pois[
            ["latitude", "longitude"]
        ].to_numpy()
    )

    tree = BallTree(
        poi_coords,
        metric="haversine"
    )

    radius_radians = (
        radius_km
        / EARTH_RADIUS_KM
    )

    neighbours = tree.query_radius(
        location_coords,
        r=radius_radians
    )

    locations[
        f"{prefix}_within_{radius_km:g}km"
    ] = [
        len(points)
        for points in neighbours
    ]

    return locations

In [47]:
test_locations = count_pois_within_radius(
    locations=test_locations,
    pois=schools,
    radius_km=2,
    prefix="schools"
)

test_locations[
    [
        "suburb",
        "nearest_school",
        "school_distance_km",
        "schools_within_2km"
    ]
]

,suburb,nearest_school,school_distance_km,schools_within_2km
0,Carlton,Carlton Gardens Primary School,0.317048,18
1,Richmond,Richmond High School,0.096545,14
2,Footscray,Footscray City Primary School,0.489763,7
3,Box Hill,Our Lady of Sion College,0.699279,13
4,Werribee,Wyndham Community and Education Centre Inc | J...,0.247769,8


## 9. Distance to Melbourne CBD

Distance to Melbourne CBD is used as a measure of accessibility to the
central employment and commercial area.

Straight-line distance is calculated first. Route-based distance will
later be calculated using OpenRouteService.

In [48]:
CBD = pd.DataFrame({
    "name": [
        "Melbourne CBD"
    ],

    "latitude": [
        -37.8136
    ],

    "longitude": [
        144.9631
    ]
})

test_locations = add_nearest_poi(
    locations=test_locations,
    pois=CBD,
    poi_name="name",
    prefix="cbd"
)

test_locations[
    [
        "suburb",
        "cbd_distance_km"
    ]
]

,suburb,cbd_distance_km
0,Carlton,1.550582
1,Richmond,3.451924
2,Footscray,5.691551
3,Box Hill,13.970995
4,Werribee,28.208879


## 10. Initial geographic feature table

The following table combines the initial geographic accessibility
features created from the external datasets.

In [49]:
test_locations[
    [
        "suburb",

        "nearest_train",
        "train_distance_km",

        "nearest_school",
        "school_distance_km",
        "schools_within_2km",

        "cbd_distance_km"
    ]
]

,suburb,nearest_train,train_distance_km,nearest_school,school_distance_km,schools_within_2km,cbd_distance_km
0,Carlton,Parkville Railway Station,0.655380,Carlton Gardens Primary School,0.317048,18,1.550582
1,Richmond,West Richmond Railway Station,0.989123,Richmond High School,0.096545,14,3.451924
2,Footscray,Footscray Railway Station,0.189086,Footscray City Primary School,0.489763,7,5.691551
3,Box Hill,Box Hill Railway Station,0.055890,Our Lady of Sion College,0.699279,13,13.970995
4,Werribee,Werribee Railway Station,0.069929,Wyndham Community and Education Centre Inc | J...,0.247769,8,28.208879


## 11. OpenRouteService setup

Straight-line distance does not account for the actual road network.
OpenRouteService (ORS) is therefore used to calculate route-based
distances to the nearest train station and Melbourne CBD.

Routing is performed at suburb level rather than for every individual
rental listing to reduce the number of API requests.

In [50]:
import os
import requests

from dotenv import load_dotenv

load_dotenv(PROJECT_ROOT / "ors_config.env")

ORS_API_KEY = os.getenv("ORS_API_KEY")

if ORS_API_KEY is None:
    raise ValueError("ORS_API_KEY was not found.")

print("ORS API key loaded successfully.")

ORS API key loaded successfully.


## 12. Route distance function

A reusable function is created to calculate driving distance between
two geographic coordinates using OpenRouteService.

OpenRouteService requires coordinates in longitude-latitude order.

In [51]:
ORS_URL = (
    "https://api.heigit.org/openrouteservice/v2/"
    "directions/driving-car"
)


def get_route_distance_km(
    origin_lon,
    origin_lat,
    destination_lon,
    destination_lat
):
    headers = {
        "Authorization": ORS_API_KEY,
        "Content-Type": "application/json"
    }

    body = {
        "coordinates": [
            [origin_lon, origin_lat],
            [destination_lon, destination_lat]
        ]
    }

    response = requests.post(
        ORS_URL,
        json=body,
        headers=headers
    )

    response.raise_for_status()

    route = response.json()

    distance_metres = (
        route["routes"][0]
        ["summary"]
        ["distance"]
    )

    return distance_metres / 1000

## 13. Test route calculation

A single route is calculated first to verify that the
OpenRouteService API connection is functioning correctly.

In [52]:
test_route = get_route_distance_km(
    origin_lon=144.967,
    origin_lat=-37.800,
    destination_lon=144.9631,
    destination_lat=-37.8136
)

print(
    f"Test route distance: "
    f"{test_route:.2f} km"
)

Test route distance: 2.02 km


## 14. Driving distance to Melbourne CBD

Driving distance to Melbourne CBD is calculated in addition to the
straight-line distance.

This provides a more realistic accessibility measure because it
accounts for the available road network.

In [53]:
CBD_LAT = -37.8136
CBD_LON = 144.9631

test_locations["cbd_route_km"] = (
    test_locations.apply(
        lambda row: get_route_distance_km(
            origin_lon=row["longitude"],
            origin_lat=row["latitude"],
            destination_lon=CBD_LON,
            destination_lat=CBD_LAT
        ),
        axis=1
    )
)

test_locations[
    [
        "suburb",
        "cbd_distance_km",
        "cbd_route_km"
    ]
]

,suburb,cbd_distance_km,cbd_route_km
0,Carlton,1.550582,2.0173
1,Richmond,3.451924,4.3447
2,Footscray,5.691551,6.6588
3,Box Hill,13.970995,14.8562
4,Werribee,28.208879,33.9019


## 15. Driving distance to nearest train station

The nearest train station is first identified using straight-line
distance. OpenRouteService is then used to calculate road distance
between each suburb location and its identified nearest station.

This two-stage approach avoids routing to every train station.

In [54]:
test_locations["train_route_km"] = (
    test_locations.apply(
        lambda row: get_route_distance_km(
            origin_lon=row["longitude"],
            origin_lat=row["latitude"],
            destination_lon=row["train_longitude"],
            destination_lat=row["train_latitude"]
        ),
        axis=1
    )
)

test_locations[
    [
        "suburb",
        "nearest_train",
        "train_distance_km",
        "train_route_km"
    ]
]

,suburb,nearest_train,train_distance_km,train_route_km
0,Carlton,Parkville Railway Station,0.655380,1.1682
1,Richmond,West Richmond Railway Station,0.989123,1.5216
2,Footscray,Footscray Railway Station,0.189086,0.1870
3,Box Hill,Box Hill Railway Station,0.055890,1.6732
4,Werribee,Werribee Railway Station,0.069929,2.3110


## 16. Geographic visualisation

A map is created to visualise the temporary suburb locations and
Victorian train stations used in the accessibility analysis.

In [55]:

from IPython.display import display

import folium

m = folium.Map(
    location=[-37.81, 144.96],
    zoom_start=9
)

for _, row in test_locations.iterrows():

    folium.CircleMarker(
        location=[
            row["latitude"],
            row["longitude"]
        ],
        radius=5,
        tooltip=row["suburb"],
        fill=True
    ).add_to(m)

for _, row in stations.iterrows():

    folium.CircleMarker(
        location=[
            row["latitude"],
            row["longitude"]
        ],
        radius=2,
        tooltip=row["stop_name"],
        fill=True
    ).add_to(m)

display(m)

## 17. Parks and open-space accessibility

The OpenStreetMap Victoria dataset will be used to extract park and
open-space locations. These will be used to construct proximity and
local amenity-count features.

In [56]:
from pyrosm import OSM

osm_path = OSM_DIR / "victoria-260908.osm.pbf"

print("OSM path:", osm_path)
print("Exists:", osm_path.exists())

osm = OSM(str(osm_path))

OSM path: ../data/raw/osm/victoria-260908.osm.pbf
Exists: True


In [57]:
park_filter = {
    "leisure": ["park", "garden", "nature_reserve"]
}

parks_gdf = osm.get_pois(
    custom_filter=park_filter
)

print("Raw park features:", len(parks_gdf))

parks_gdf.head()

# Remove features without geometry
parks_gdf = parks_gdf[
    parks_gdf.geometry.notna()
].copy()

# Project to a metric CRS before calculating centroids
parks_projected = parks_gdf.to_crs(epsg=7855)

# Represent each park by its centroid
parks_projected["geometry"] = (
    parks_projected.geometry.centroid
)

# Convert back to latitude/longitude
parks_points = parks_projected.to_crs(epsg=4326)

parks = pd.DataFrame({
    "park_name": parks_points["name"],
    "latitude": parks_points.geometry.y,
    "longitude": parks_points.geometry.x
})

parks["park_name"] = parks["park_name"].fillna(
    "Unnamed park"
)

parks = parks.dropna(
    subset=["latitude", "longitude"]
).reset_index(drop=True)

print("Usable park records:", len(parks))

parks.head()

Raw park features: 25553
Usable park records: 25553


,park_name,latitude,longitude
0,Unnamed park,-37.832640,145.021231
1,Katandra Football Netball Club,-36.228082,145.560527
2,Websters Reserve,-37.747787,145.155588
3,Number Two Creek Reserve,-37.523376,145.355380
4,Azalea Garden,-37.547945,143.821110


In [58]:
print("Park records before removing duplicates:", len(parks))

print(
    "Unique park coordinates:",
    parks[["latitude", "longitude"]]
    .drop_duplicates()
    .shape[0]
)

print(
    "Unique named parks:",
    parks.loc[
        parks["park_name"] != "Unnamed park",
        "park_name"
    ].nunique()
)

parks = (
    parks
    .drop_duplicates(
        subset=["latitude", "longitude"]
    )
    .reset_index(drop=True)
)

print("Park records after removing duplicates:", len(parks))

Park records before removing duplicates: 25553
Unique park coordinates: 25551
Unique named parks: 10630
Park records after removing duplicates: 25551


In [59]:
test_locations = add_nearest_poi(
    test_locations,
    parks,
    poi_name="park_name",
    prefix="park"
)

test_locations[
    ["suburb", "nearest_park", "park_distance_km"]
]

,suburb,nearest_park,park_distance_km
0,Carlton,La Mama Square,0.132489
1,Richmond,Citizens Park,0.179744
2,Footscray,Railway Reserve,0.072492
3,Box Hill,Unnamed park,0.046359
4,Werribee,Unnamed park,0.083071


In [60]:
test_locations = count_pois_within_radius(
    test_locations,
    parks,
    radius_km=2,
    prefix="parks"
)

test_locations[
    [
        "suburb",
        "nearest_park",
        "park_distance_km",
        "parks_within_2km"
    ]
]

,suburb,nearest_park,park_distance_km,parks_within_2km
0,Carlton,La Mama Square,0.132489,431
1,Richmond,Citizens Park,0.179744,204
2,Footscray,Railway Reserve,0.072492,152
3,Box Hill,Unnamed park,0.046359,101
4,Werribee,Unnamed park,0.083071,81


## 18. Shopping and amenity accessibility

OpenStreetMap points of interest will be used to identify shopping
and retail amenities. Distance and nearby-amenity counts will be
constructed as additional liveability features.

In [61]:
#limit shops to malls, department stores, and supermarkets only for now
shopping_filter = {
    "shop": ["mall", "department_store", "supermarket"]
}

shopping_gdf = osm.get_pois(
    custom_filter=shopping_filter
)

print("Shopping features:", len(shopping_gdf))
shopping_gdf.head()

Shopping features: 2085


,version,visible,tags,lat,id,changeset,lon,timestamp,addr:country,addr:housenumber,...,operator,phone,ref,website,organic,second_hand,shop,geometry,osm_type,url
0,3,False,NaN,-37.277362,32193447,0.0,144.733861,1361574238,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,supermarket,POINT (144.73386 -37.27736),node,NaN
1,15,False,"{""brand"":""Coles"",""brand:wikidata"":""Q1108172"",""...",-37.791434,218028470,0.0,145.171842,1774608595,NaN,55,...,Coles Group,NaN,NaN,https://www.coles.com.au/,NaN,NaN,supermarket,POINT (145.17184 -37.79143),node,NaN
2,14,False,"{""brand"":""Coles"",""brand:wikidata"":""Q1108172"",""...",-37.868257,260559651,0.0,145.240404,1786865274,NaN,NaN,...,Coles Group,NaN,NaN,https://www.coles.com.au/,NaN,NaN,supermarket,POINT (145.2404 -37.86826),node,NaN
3,12,False,"{""addr:state"":""VIC"",""addr:suburb"":""Forest Hill...",-37.834987,260710476,0.0,145.164618,1693297610,NaN,270,...,NaN,+61 3 8878 1300,NaN,https://www.target.com.au/store/vic/forest-hil...,NaN,NaN,department_store,POINT (145.16462 -37.83499),node,NaN
4,8,False,"{""brand"":""Coles"",""brand:wikidata"":""Q1108172"",""...",-37.834960,260710477,0.0,145.164335,1774608595,NaN,NaN,...,Coles Group,NaN,NaN,https://www.coles.com.au/,NaN,NaN,supermarket,POINT (145.16433 -37.83496),node,NaN


In [62]:
shopping_gdf = shopping_gdf[
    shopping_gdf.geometry.notna()
].copy()

shopping_projected = shopping_gdf.to_crs(epsg=7855)

shopping_projected["geometry"] = (
    shopping_projected.geometry.centroid
)

shopping_points = shopping_projected.to_crs(epsg=4326)

shopping = pd.DataFrame({
    "shopping_name": shopping_points["name"],
    "latitude": shopping_points.geometry.y,
    "longitude": shopping_points.geometry.x
})

shopping["shopping_name"] = (
    shopping["shopping_name"]
    .fillna("Unnamed shop")
)

shopping = shopping.dropna(
    subset=["latitude", "longitude"]
).reset_index(drop=True)

print("Usable shopping locations:", len(shopping))

shopping.head()

Usable shopping locations: 2085


,shopping_name,latitude,longitude
0,IGA,-37.277362,144.733861
1,Coles,-37.791434,145.171842
2,Coles,-37.868258,145.240404
3,Target,-37.834987,145.164618
4,Coles,-37.834960,145.164335


In [63]:
test_locations = add_nearest_poi(
    test_locations,
    shopping,
    poi_name="shopping_name",
    prefix="shopping"
)

test_locations = count_pois_within_radius(
    test_locations,
    shopping,
    radius_km=2,
    prefix="shopping"
)

test_locations[
    [
        "suburb",
        "nearest_shopping",
        "shopping_distance_km",
        "shopping_within_2km"
    ]
]

,suburb,nearest_shopping,shopping_distance_km,shopping_within_2km
0,Carlton,IGA,0.147293,72
1,Richmond,Coles,0.283312,18
2,Footscray,Western Halal Meat,0.140064,18
3,Box Hill,Goldplus Supermarket,0.068849,17
4,Werribee,Woolworths,0.184393,8


## 19. Initial geospatial feature table

The constructed features combine public transport, education and
central-city accessibility measures.

The temporary suburb locations will later be replaced by locations
derived from the group's complete rental-property dataset.

In [64]:
geo_features = test_locations[
    [
        "suburb",

        "nearest_train",
        "train_distance_km",
        "train_route_km",

        "nearest_school",
        "school_distance_km",
        "schools_within_2km",

        "nearest_park",
        "park_distance_km",
        "parks_within_2km",

        "nearest_shopping",
        "shopping_distance_km",
        "shopping_within_2km",

        "cbd_distance_km",
        "cbd_route_km"
    ]
].copy()

geo_features

,suburb,nearest_train,train_distance_km,train_route_km,nearest_school,school_distance_km,schools_within_2km,nearest_park,park_distance_km,parks_within_2km,nearest_shopping,shopping_distance_km,shopping_within_2km,cbd_distance_km,cbd_route_km
0,Carlton,Parkville Railway Station,0.655380,1.1682,Carlton Gardens Primary School,0.317048,18,La Mama Square,0.132489,431,IGA,0.147293,72,1.550582,2.0173
1,Richmond,West Richmond Railway Station,0.989123,1.5216,Richmond High School,0.096545,14,Citizens Park,0.179744,204,Coles,0.283312,18,3.451924,4.3447
2,Footscray,Footscray Railway Station,0.189086,0.1870,Footscray City Primary School,0.489763,7,Railway Reserve,0.072492,152,Western Halal Meat,0.140064,18,5.691551,6.6588
3,Box Hill,Box Hill Railway Station,0.055890,1.6732,Our Lady of Sion College,0.699279,13,Unnamed park,0.046359,101,Goldplus Supermarket,0.068849,17,13.970995,14.8562
4,Werribee,Werribee Railway Station,0.069929,2.3110,Wyndham Community and Education Centre Inc | J...,0.247769,8,Unnamed park,0.083071,81,Woolworths,0.184393,8,28.208879,33.9019


In [65]:
geo_features.info()
geo_features.isna().sum()
geo_features.describe()

<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   suburb                5 non-null      str    
 1   nearest_train         5 non-null      str    
 2   train_distance_km     5 non-null      float64
 3   train_route_km        5 non-null      float64
 4   nearest_school        5 non-null      str    
 5   school_distance_km    5 non-null      float64
 6   schools_within_2km    5 non-null      int64  
 7   nearest_park          5 non-null      str    
 8   park_distance_km      5 non-null      float64
 9   parks_within_2km      5 non-null      int64  
 10  nearest_shopping      5 non-null      str    
 11  shopping_distance_km  5 non-null      float64
 12  shopping_within_2km   5 non-null      int64  
 13  cbd_distance_km       5 non-null      float64
 14  cbd_route_km          5 non-null      float64
dtypes: float64(7), int64(3), str(5)
memory

,train_distance_km,train_route_km,school_distance_km,schools_within_2km,park_distance_km,parks_within_2km,shopping_distance_km,shopping_within_2km,cbd_distance_km,cbd_route_km
count,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000
mean,0.391882,1.372200,0.370081,12.000000,0.102831,193.800000,0.164782,26.600000,10.574786,12.355780
std,0.413461,0.781118,0.231996,4.527693,0.053140,140.935091,0.078349,25.725474,10.937469,12.980799
min,0.055890,0.187000,0.096545,7.000000,0.046359,81.000000,0.068849,8.000000,1.550582,2.017300
25%,0.069929,1.168200,0.247769,8.000000,0.072492,101.000000,0.140064,17.000000,3.451924,4.344700
50%,0.189086,1.521600,0.317048,13.000000,0.083071,152.000000,0.147293,18.000000,5.691551,6.658800
75%,0.655380,1.673200,0.489763,14.000000,0.132489,204.000000,0.184393,18.000000,13.970995,14.856200
max,0.989123,2.311000,0.699279,18.000000,0.179744,431.000000,0.283312,72.000000,28.208879,33.901900


## 20. Assumptions and limitations

- The current analysis uses temporary suburb coordinates for pipeline
  development. These will be replaced with locations derived from the
  complete rental-property dataset.

- Straight-line distance provides a computationally efficient baseline
  but does not represent actual travel distance.

- OpenRouteService driving distance accounts for the road network but
  does not necessarily represent public-transport travel time.

- Route calculations will be performed at suburb level rather than for
  every individual property to reduce API usage. This means variation
  between properties within the same suburb may not be captured.

- School accessibility is represented using nearest-school distance and
  the number of schools within 2 km. The 2 km radius is a modelling
  assumption and may not represent the appropriate accessibility
  threshold for every household.

- Amenity measures depend on the completeness and accuracy of the
  external geographic datasets.